In [ ]:
print("start")
from comet_ml import Experiment
import torch
import os
import torch.nn as nn
import torch.nn.functional as F
from datetime import datetime
data = torch.load("/home/jovyan/lab3/coco_train_preprocessed.pt")
print("klar")


# Additive model (bigger + dropout)
print("Additive Upgraded start")

run_name = datetime.now().strftime("additive-upgraded-%Y%m%d-%H%M%S")
print(run_name)

# load the preprocessed dataset
samples = data["samples"]
vocab = data["vocab"]
idx_to_word = data["idx_to_word"]
max_length = data["max_length"]

class ImageCaptionModel(nn.Module):
    def __init__(self, vocab_size, max_length):
        super(ImageCaptionModel, self).__init__()

        # 25k to 512
        self.image_embedding = nn.Sequential(
            nn.Linear(25088, 512),
            nn.ReLU(),
            nn.Dropout(0.3) 
        )

    
        self.caption_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=512
        )
        
        # Added Dropout to LSTM to prevent the "staircase" memorization effect
        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=512,
            batch_first=True
        )
        self.lstm_dropout = nn.Dropout(0.5)

        # Output layer
        self.output_layer = nn.Linear(512, vocab_size)

    def forward(self, image_features, captions):
        #to make sure the embeddings are the same size for the additive combination
        img_emb = self.image_embedding(image_features)                      # (batch, 512)
        cap_emb = self.caption_embedding(captions)                          # (batch, seq_len, 512)
        
        lstm_out, _ = self.lstm(cap_emb)                                    # (batch, seq_len, 512)
        lstm_out = self.lstm_dropout(lstm_out)                              # (batch, seq_len, 512)

        # Expand image embedding across all timesteps and add
        img_expanded = img_emb.unsqueeze(1).repeat(1, lstm_out.size(1), 1)  # (batch, seq_len, 512)
        combined = lstm_out + img_expanded                                  # (batch, seq_len, 512)

        output = self.output_layer(combined)                                # (batch, seq_len, vocab_size)
        return output

# Model Initialization
model = ImageCaptionModel(
    vocab_size=len(vocab),
    max_length=max_length
)

# Loss function
criterion = nn.CrossEntropyLoss(
    ignore_index=vocab["<pad>"]
)

# Optimizer
# Weight decay helps regularize the large linear layer (25088 -> 512)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0003,
    weight_decay=1e-5
)

# Comet ML experiment
experiment = Experiment(
    api_key        = "lZ6VM5qo8Rz8JKYXjXVS3zWzs",
    project_name   = "D7047E/Lab3",
    experiment_name = run_name,
)

experiment.log_parameters({
    "model":         "additive_upgraded",
    "vocab_size":    len(vocab),
    "max_length":    max_length,
    "embed_dim":     512,
    "hidden_size":   512,
    "dropout":       0.5,
    "learning_rate": 0.0003,
    "combination":   "addition",
    "clipping":      1.0,
    "name":          run_name
})

print(model)

In [ ]:
#Training (Additive)
import os         
import torch    
from torch.utils.data import Dataset, DataLoader 
from tqdm import tqdm  


TRAIN_FEATURES_PATH = "/home/jovyan/annabell/train_features.pt"      
VAL_FEATURES_PATH = "/home/jovyan/annabell/val_features.pt"          
VAL_PREPROCESSED_PATH = "/home/jovyan/lab3/coco_val_preprocessed_new.pt" 



BATCH_SIZE = 64    
NUM_EPOCHS = 5     
NUM_WORKERS = 4   
# class to connect features and tokenized captions
class CaptionDataset(Dataset):
    def __init__(self, samples, feature_dict):
        # lookup for base filename (miscommunication between feature and preprocessing)
        self.feature_lookup = {
            os.path.basename(k): v
            for k, v in feature_dict.items()
        }

        # keep only samples whose image file has a corresponding feature in the dict
        self.samples = [
            s for s in samples
            if os.path.basename(s["image_path"]) in self.feature_lookup
        ]

        # skip missing features
        skipped = len(samples) - len(self.samples)
        if skipped:
            print(f"  Warning: skipped {skipped} samples with no matching feature.")

    def __len__(self):
        return len(self.samples)   # total number of usable (image, caption) pairs

    def __getitem__(self, idx):
        sample = self.samples[idx]                             # retrieve the idx-th sample dict
        key = os.path.basename(sample["image_path"])          # extract bare filename to use as lookup key
        image_feature = self.feature_lookup[key].float()      # fetch the pre-computed image embedding, cast to float32

        token_ids = sample["token_ids"]      # full sequence of token IDs including <start> and <end>
        caption_input = token_ids[:-1]       # input to the decoder: all tokens except the last (starts with <start>)
        caption_target = token_ids[1:]       # target for loss: all tokens except the first (ends with <end>)

        return image_feature, caption_input, caption_target   # return the three tensors as one training example



# CAPTION GENERATION
def generate_caption(model, image_feature, vocab, idx_to_word, max_length=30, device="cpu"):
    model.eval()                           # switch model to inference mode (disables dropout etc.)
    with torch.no_grad():                  # disable gradient computation to save memory and speed up inference
        img = image_feature.unsqueeze(0).to(device)   # add a batch dimension → shape (1, feature_dim), move to device
        img_emb = model.image_embedding(img)           # project raw image feature into the model's embedding space

        input_token = torch.tensor([[vocab["<start>"]]], device=device)  # seed the decoder with the <start> token
        hidden = None   # LSTM hidden state starts as None
        words = []      # accumulate decoded words here

        for _ in range(max_length):   # generate at most max_length words
            cap_emb = model.caption_embedding(input_token)       # embed the current input token
            lstm_out, hidden = model.lstm(cap_emb, hidden)       # run one LSTM step; update hidden state
            combined = lstm_out + img_emb.unsqueeze(1)           # fuse language and image signals by addition
            logits = model.output_layer(combined)                # project to vocabulary logits

            next_token = logits.argmax(dim=-1)                   # greedy pick: take the highest-scoring token
            word = idx_to_word.get(next_token.item(), "<unk>")   # map token ID to word string

            if word == "<end>":           # stop decoding when the model produces the end-of-sequence token
                break
            if word not in ("<start>", "<pad>"):  # skip any special/padding tokens that aren't real words
                words.append(word)        # add the predicted word to the output sequence

            input_token = next_token      # feed the predicted token back in as the next input (autoregressive)

    return " ".join(words)   # join all collected words into a single caption string



# TRAINING LOOP

def train_epoch(model, dataloader, criterion, optimizer, device, epoch):
    model.train()    # switch model to training mode (enables dropout, batch norm updates, etc.)
    total_loss = 0   # accumulator for summing batch losses across the epoch

    for batch_idx, (image_features, caption_inputs, caption_targets) in enumerate(tqdm(dataloader, desc=f"Epoch {epoch}")):
        image_features  = image_features.to(device)    # move image embeddings to GPU/CPU
        caption_inputs  = caption_inputs.to(device)    # move decoder input tokens to GPU/CPU
        caption_targets = caption_targets.to(device)   # move target tokens to GPU/CPU

        optimizer.zero_grad()   # clear gradients from the previous step to prevent accumulation

        outputs = model(image_features, caption_inputs)          # forward pass: get logit predictions for each timestep
        outputs_flat = outputs.reshape(-1, outputs.size(-1))     # flatten (batch, seq_len, vocab) → (batch*seq_len, vocab)
        targets_flat = caption_targets.reshape(-1)               # flatten (batch, seq_len) → (batch*seq_len,) for CrossEntropyLoss

        loss = criterion(outputs_flat, targets_flat)   # compute cross-entropy loss comparing predictions to targets
        loss.backward()                                # backpropagate: compute gradients through the model
        optimizer.step()                               # update model weights using the computed gradients

        total_loss += loss.item()   # accumulate the scalar loss value (detached from the computation graph)

        # log the batch-level loss to the experiment tracker every 100 batches
        if batch_idx % 100 == 0:
            experiment.log_metric("batch_loss", loss.item(), step=epoch * len(dataloader) + batch_idx)

    avg_loss = total_loss / len(dataloader)                       # compute mean loss over all batches in the epoch
    print(f"  Epoch {epoch} — train loss: {avg_loss:.4f}")        # print the epoch summary
    experiment.log_metric("train_loss", avg_loss, step=epoch)    # log epoch-level train loss to experiment tracker
    return avg_loss   # return average loss so the caller can track progress



# EVAL LOOP

def eval_epoch(model, dataloader, criterion, device, epoch):
    model.eval()     # switch to eval mode (disables dropout, freezes batch norm, etc.)
    total_loss = 0   # accumulator for validation loss

    with torch.no_grad():   # no gradients needed during evaluation — saves memory and speeds things up
        for image_features, caption_inputs, caption_targets in tqdm(dataloader, desc="Evaluating"):
            image_features  = image_features.to(device)    # move batch to target device
            caption_inputs  = caption_inputs.to(device)
            caption_targets = caption_targets.to(device)

            outputs = model(image_features, caption_inputs)         # forward pass only (no backward)
            outputs_flat = outputs.reshape(-1, outputs.size(-1))    # flatten predictions for loss computation
            targets_flat = caption_targets.reshape(-1)              # flatten targets to match

            loss = criterion(outputs_flat, targets_flat)   # compute loss (same criterion as training)
            total_loss += loss.item()                      # accumulate batch loss

    avg_loss = total_loss / len(dataloader)                      # mean validation loss across all batches
    print(f"  Epoch {epoch} — val loss:   {avg_loss:.4f}")       # print val summary
    experiment.log_metric("val_loss", avg_loss, step=epoch)      # log to experiment tracker
    return avg_loss   # return for tracking and comparison with training loss



# MAIN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # use GPU if available, otherwise CPU
model.to(device)          # move model parameters to the selected device
print(f"Device: {device}")

# Load train features and build train dataset
print("\nLoading train image features...")
train_features = torch.load(TRAIN_FEATURES_PATH, weights_only=False)  # deserialise the pre-computed image feature dict
print(f"  {len(train_features)} train image features")

print("\nBuilding train dataset...")
train_dataset = CaptionDataset(samples, train_features)   # pair captions (in 'samples') with their image features
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,                      # how many samples per batch
    shuffle=True,                               # randomise order each epoch to improve training
    num_workers=NUM_WORKERS,                    # use multiple CPU processes to prefetch data
    pin_memory=(device.type == "cuda")          # pin memory for faster CPU→GPU transfers (only useful with CUDA)
)
print(f"  {len(train_dataset)} samples ready for training")

# Load val features and build val dataset
print("\nLoading val image features...")
val_features = torch.load(VAL_FEATURES_PATH, weights_only=False)   # load validation image embeddings
print(f"  {len(val_features)} val image features")

print("\nBuilding val dataset...")
val_data = torch.load(VAL_PREPROCESSED_PATH, weights_only=False)   # load preprocessed val samples (captions + token IDs)
val_dataset = CaptionDataset(val_data["samples"], val_features)     # build val dataset the same way as train
val_dataloader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,                              # no shuffling for validation — order doesn't matter for evaluation
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)
print(f"  {len(val_dataset)} samples ready for evaluation")

# Training
print("\nStarting training...")
loss_history = []   # list to store per-epoch train/val losses for later inspection or plotting
for epoch in range(1, NUM_EPOCHS + 1):   # iterate over epochs 1 through NUM_EPOCHS inclusive
    avg_train_loss = train_epoch(model, train_dataloader, criterion, optimizer, device, epoch)  # run one training epoch
    avg_val_loss   = eval_epoch(model, val_dataloader, criterion, device, epoch)                # evaluate on validation set
    loss_history.append({"train": avg_train_loss, "val": avg_val_loss})   # record both losses for this epoch

    # generate a sample caption from the first training image to qualitatively track model improvement
    img_feat = train_dataset[0][0]   # grab the image feature from the first training sample
    caption  = generate_caption(model, img_feat, vocab, idx_to_word, max_length, device)  # decode a caption greedily
    ref      = train_dataset.samples[0]["caption"]   # the ground-truth reference caption for comparison
    print(f"  Generated: {caption}")   # print model's output
    print(f"  Reference: {ref}")       # print human-written reference for side-by-side comparison

# Save model using run_name
model_save_path = f"./{run_name}.pt"   # build save path from the experiment's run name
torch.save({
    "model_state_dict": model.state_dict(),   # the learned weights
    "vocab":            vocab,                # word → index mapping (needed for inference)
    "idx_to_word":      idx_to_word,          # index → word mapping (needed to decode predictions)
    "max_length":       max_length,           # max caption length used during training
    "loss_history":     loss_history          # epoch-by-epoch loss log for analysis
}, model_save_path)
print(f"\nModel saved to {model_save_path}")

# Show 5 captions on val images as a final qualitative check
print("\n-- Sample captions on val images --")
for key in list(val_features.keys())[:5]:    # iterate over the first 5 validation image keys
    feat    = val_features[key].float()      # retrieve the image feature and cast to float32
    caption = generate_caption(model, feat, vocab, idx_to_word, max_length, device)  # generate a caption
    print(f"  {key}: {caption}")             # print the filename alongside its generated caption

experiment.end()   # signal to the experiment tracker (e.g. Comet ML) that the run is complete

In [ ]:
#Attention model
class ImageCaptionModel(nn.Module):  # inherit from PyTorch's base class for all neural networks
    def __init__(self, vocab_size, max_length):  # vocab_size determines output layer size; max_length is unused here but kept for interface consistency
        super(ImageCaptionModel, self).__init__()  # call the parent nn.Module constructor — required boilerplate
        #MLP
        self.image_embedding = nn.Sequential(
            nn.Linear(25088, 512),  # learned projection from VGG's flattened feature vector to 512-dim space
            nn.ReLU(),              # non-linearity so the projection can learn more than just a linear mapping
            nn.Dropout(0.3)         # randomly zero 30% of units during training to reduce overfitting
        )

        self.caption_embedding = nn.Embedding(
            num_embeddings=vocab_size,  # one row per word in the vocabulary
            embedding_dim=512           # each word maps to a 512-dim vector, matching the image embedding size
        )

        self.lstm = nn.LSTM(
            input_size=512,    # expects 512-dim input vectors (matching the caption embeddings)
            hidden_size=512,   # internal hidden state is also 512-dim
            batch_first=True   # input/output tensors are (batch, seq_len, features) rather than (seq_len, batch, features)
        )
        self.lstm_dropout = nn.Dropout(0.5)  # drop 50% of LSTM outputs during training to prevent overfitting

        # attention layer that lets each LSTM output token "look at" the image and decide what's relevant
        self.attention = nn.MultiheadAttention(
            embed_dim=512,     # all queries, keys, and values are 512-dim
            num_heads=8,       # split into 8 parallel attention heads, each working in 64-dim subspace
            batch_first=True   # same batch-first convention as the LSTM
        )

        self.output_layer = nn.Linear(512, vocab_size)  # project the final 512-dim representation to a score for every word in the vocabulary

    def forward(self, image_features, captions):
        img_emb = self.image_embedding(image_features)  # compress raw image features → (batch, 512)
        cap_emb = self.caption_embedding(captions)      # convert token ID sequence → (batch, seq_len, 512)

        h0 = img_emb.unsqueeze(0)   # reshape image embedding to be the LSTM's initial hidden state → (1, batch, 512); this is how image context is injected into the sequence model
        c0 = torch.zeros_like(h0)   # initial cell state set to zeros — no prior memory, only the hidden state carries the image signal

        lstm_out, _ = self.lstm(cap_emb, (h0, c0))  # run the full caption sequence through the LSTM, seeded with the image; output is (batch, seq_len, 512)
        lstm_out = self.lstm_dropout(lstm_out)       # apply dropout to LSTM outputs before attention

        img_context = img_emb.unsqueeze(1)  # reshape image embedding to a single-token sequence → (batch, 1, 512), so attention can treat it as one key/value "slot"

        attn_output, _ = self.attention(
            query=lstm_out,       # each caption position asks a question: "given what I've generated so far, what part of the image matters?"
            key=img_context,      # the image is the only key — attention scores are computed against it
            value=img_context     # the image is also the value — what gets retrieved when attention fires
        )
        # attn_output shape: (batch, seq_len, 512) — one image-informed vector per caption position

        combined = lstm_out + attn_output  # add the language signal and the image-attention signal together (residual-style fusion)

        return self.output_layer(combined)  # project fused representation to vocabulary logits → (batch, seq_len, vocab_size)
        
        """
        #To compare against additive
    def forward(self, image_features, captions):
        img_emb = self.image_embedding(image_features)                      # (batch, 512)
        cap_emb = self.caption_embedding(captions)                          # (batch, seq_len, 512)
        lstm_out, _ = self.lstm(cap_emb)                                    # (batch, seq_len, 512)
        lstm_out = self.lstm_dropout(lstm_out)                              # (batch, seq_len, 512)

        # Expand image embedding across all timesteps and add
        img_expanded = img_emb.unsqueeze(1).repeat(1, lstm_out.size(1), 1)  # (batch, seq_len, 512)
        combined = lstm_out + img_expanded                                  # (batch, seq_len, 512)

        output = self.output_layer(combined)                                # (batch, seq_len, vocab_size)
        return output
        """

def generate_caption(model, image_feature, vocab, idx_to_word, max_length=30, device="cpu"):
    model.eval()
    with torch.no_grad():
        img = image_feature.unsqueeze(0).to(device)
        img_emb = model.image_embedding(img)
        img_context = img_emb.unsqueeze(1)

        h = img_emb.unsqueeze(0) 
        c = torch.zeros_like(h)
        hidden = (h, c)

        input_token = torch.tensor([[vocab["<start>"]]], device=device)
        words = []

        for _ in range(max_length):
            cap_emb = model.caption_embedding(input_token)
            lstm_out, hidden = model.lstm(cap_emb, hidden)
            
            attn_out, _ = model.attention(query=lstm_out, key=img_context, value=img_context)
            logits = model.output_layer(lstm_out + attn_out)

            next_token = logits.argmax(dim=-1)
            word = idx_to_word.get(next_token.item(), "<unk>")

            if word == "<end>": break
            if word not in ("<start>", "<pad>"): words.append(word)
            input_token = next_token

    return " ".join(words)

In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    # ── Load model ─────────────────────────────
    checkpoint = torch.load(MODEL_PATH, map_location=device)

    vocab = checkpoint["vocab"]
    idx_to_word = {int(k): v for k, v in checkpoint["idx_to_word"].items()}
    max_length = checkpoint["max_length"]

    model = ImageCaptionModel(len(vocab), max_length).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # ── Load data ──────────────────────────────
    val_features = torch.load(VAL_FEATURES_PATH)
    val_data = torch.load(VAL_PREPROCESSED_PATH)
    val_samples = val_data["samples"]

    feature_lookup = {os.path.basename(k): v for k, v in val_features.items()}

    # ── CLEAN REFERENCES ───────────────────────
    refs_by_image = defaultdict(list)

    for s in val_samples:
        key = os.path.basename(s["image_path"])
        cap = s["caption"]
        if isinstance(cap, dict):
            cap = cap.get("caption", "")
        refs_by_image[key].append(str(cap))

    eval_keys = [k for k in refs_by_image if k in feature_lookup]
    print("Images:", len(eval_keys))

    # ── Generate captions ──────────────────────
    hypotheses = {}
    references = {}

    for k in tqdm(eval_keys, desc="Generating Captions"):
        feat = feature_lookup[k].float()
        hypotheses[k] = generate_caption(
            model, feat, vocab, idx_to_word, max_length, device
        )
        references[k] = refs_by_image[k]

    # ── BLEU ───────────────────────────────────
    print("\nBLEU:")
    smoother = SmoothingFunction().method1
    bleu_refs = [[r.split() for r in references[k]] for k in eval_keys]
    bleu_hyps = [hypotheses[k].split() for k in eval_keys]

    print("BLEU-1:", corpus_bleu(bleu_refs, bleu_hyps, weights=(1,0,0,0), smoothing_function=smoother))
    print("BLEU-2:", corpus_bleu(bleu_refs, bleu_hyps, weights=(0.5,0.5,0,0), smoothing_function=smoother))
    print("BLEU-3:", corpus_bleu(bleu_refs, bleu_hyps, weights=(0.33,0.33,0.33,0), smoothing_function=smoother))
    print("BLEU-4:", corpus_bleu(bleu_refs, bleu_hyps, weights=(0.25,0.25,0.25,0.25), smoothing_function=smoother))

    # ── METEOR ─────────────────────────────────
    print("\nMETEOR:")
    meteor_scores = [
        meteor_score([r.split() for r in references[k]], hypotheses[k].split())
        for k in eval_keys
    ]
    print(sum(meteor_scores) / len(meteor_scores))

    # ── ROUGE ──────────────────────────────────
    print("\nROUGE:")
    r_scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1, r2, rL = [], [], []

    for k in eval_keys:
        best1 = best2 = bestL = 0
        for ref in references[k]:
            scores = r_scorer.score(ref, hypotheses[k])
            best1 = max(best1, scores["rouge1"].fmeasure)
            best2 = max(best2, scores["rouge2"].fmeasure)
            bestL = max(bestL, scores["rougeL"].fmeasure)
        r1.append(best1)
        r2.append(best2)
        rL.append(bestL)

    print("ROUGE-1:", sum(r1) / len(r1))
    print("ROUGE-2:", sum(r2) / len(r2))
    print("ROUGE-L:", sum(rL) / len(rL))

    # ── CIDEr ──────────────────────────────────
    print("\nCIDEr:")
    gts = {k: references[k] for k in eval_keys}
    res = {k: [hypotheses[k]] for k in eval_keys}
    cider = Cider()
    score, _ = cider.compute_score(gts, res)
    print(score)


if __name__ == "__main__":
    main()